In [ ]:
# Cell 1: Load the processed UK, France, and Ethiopia datasets and build a harmonized three-country crash-severity summary.

import pandas as pd
import numpy as np

uk_df = pd.read_csv(
    "../data/processed/uk_processed.csv"
)

france_df = pd.read_csv(
    "../data/processed/france_processed.csv"
)

ethiopia_df = pd.read_csv(
    "../data/processed/ethiopia_processed.csv"
)

countries = {
    "UK": uk_df,
    "France": france_df,
    "Ethiopia": ethiopia_df
}

country_summary = []

for country, df in countries.items():

    counts = (
        df["severity_class"]
        .value_counts()
        .reindex(
            ["Slight", "Serious", "Fatal"],
            fill_value=0
        )
    )

    total = len(df)

    slight = counts["Slight"]
    serious = counts["Serious"]
    fatal = counts["Fatal"]
    severe = serious + fatal

    country_summary.append({
        "Country": country,
        "Total_Crashes": total,

        "Slight_n": slight,
        "Slight_%": slight / total * 100,

        "Serious_n": serious,
        "Serious_%": serious / total * 100,

        "Fatal_n": fatal,
        "Fatal_%": fatal / total * 100,

        "Severe_n": severe,
        "Severe_%": severe / total * 100
    })

country_summary_df = pd.DataFrame(
    country_summary
)

percentage_columns = [
    "Slight_%",
    "Serious_%",
    "Fatal_%",
    "Severe_%"
]

country_summary_df[
    percentage_columns
] = (
    country_summary_df[
        percentage_columns
    ].round(2)
)

print("Dataset shapes:")
print("UK:", uk_df.shape)
print("France:", france_df.shape)
print("Ethiopia:", ethiopia_df.shape)

print("\nCross-Country Crash Severity Summary")
print("=" * 100)

print(
    country_summary_df.to_string(
        index=False
    )
)

In [ ]:
# Cell 2: Compare the prevalence of Slight, Serious, Fatal, and combined Severe crashes across the three countries.

severity_comparison = country_summary_df[
    [
        "Country",
        "Slight_%",
        "Serious_%",
        "Fatal_%",
        "Severe_%"
    ]
].copy()

severity_comparison = severity_comparison.set_index("Country")

print("Cross-Country Severity Prevalence (%)")
print("=" * 70)
print(severity_comparison.to_string())

print("\nPairwise absolute differences in Fatal prevalence (percentage points):")

countries_list = severity_comparison.index.tolist()

for i in range(len(countries_list)):
    for j in range(i + 1, len(countries_list)):
        c1 = countries_list[i]
        c2 = countries_list[j]

        diff = (
            severity_comparison.loc[c1, "Fatal_%"]
            - severity_comparison.loc[c2, "Fatal_%"]
        )

        print(
            f"{c1} vs {c2}: {diff:+.2f} percentage points"
        )

print("\nPairwise absolute differences in Severe prevalence (percentage points):")

for i in range(len(countries_list)):
    for j in range(i + 1, len(countries_list)):
        c1 = countries_list[i]
        c2 = countries_list[j]

        diff = (
            severity_comparison.loc[c1, "Severe_%"]
            - severity_comparison.loc[c2, "Severe_%"]
        )

        print(
            f"{c1} vs {c2}: {diff:+.2f} percentage points"
        )

In [ ]:
# Cell 3: Verify the harmonized predictors available across UK, France, and Ethiopia and inspect cross-country category compatibility.

common_features_3countries = [
    "hour",
    "day_common",
    "weather_common",
    "light_common",
    "surface_common",
    "junction_common",
    "vehicle_count",
    "has_motorcycle",
    "has_heavy_vehicle",
    "has_public_transport",
    "has_two_wheeler"
]

print("Common feature availability")
print("=" * 80)

for country, df in countries.items():

    missing = [
        col for col in common_features_3countries
        if col not in df.columns
    ]

    print(f"\n{country}")
    print("Missing common features:", missing)
    print(
        "Available common features:",
        len(common_features_3countries) - len(missing),
        "/",
        len(common_features_3countries)
    )


print("\n\nCross-country data types and unique-value counts")
print("=" * 80)

feature_compatibility = []

for feature in common_features_3countries:

    row = {
        "Feature": feature
    }

    for country, df in countries.items():

        if feature in df.columns:

            row[f"{country}_dtype"] = str(df[feature].dtype)

            row[f"{country}_unique_n"] = (
                df[feature]
                .nunique(dropna=True)
            )

            row[f"{country}_missing_n"] = (
                df[feature]
                .isna()
                .sum()
            )

        else:

            row[f"{country}_dtype"] = "Missing"
            row[f"{country}_unique_n"] = np.nan
            row[f"{country}_missing_n"] = np.nan

    feature_compatibility.append(row)


feature_compatibility_df = pd.DataFrame(
    feature_compatibility
)

print(
    feature_compatibility_df.to_string(
        index=False
    )
)


print("\n\nCategorical value comparison")
print("=" * 80)

categorical_common = [
    "day_common",
    "weather_common",
    "light_common",
    "surface_common",
    "junction_common"
]

for feature in categorical_common:

    print("\n" + "-" * 80)
    print("Feature:", feature)

    for country, df in countries.items():

        values = sorted(
            df[feature]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        print(f"{country}: {values}")

In [ ]:
# Cell 4: Quantify cross-country distribution shifts in all 11 harmonized predictors using normalized distributions and Jensen-Shannon distance.

from scipy.spatial.distance import jensenshannon

pairwise_countries = [
    ("UK", "France"),
    ("UK", "Ethiopia"),
    ("France", "Ethiopia")
]

distribution_shift_results = []

for feature in common_features_3countries:

    # Treat every feature as discrete for distribution comparison
    all_values = sorted(
        set(
            countries["UK"][feature].astype(str).unique()
        )
        | set(
            countries["France"][feature].astype(str).unique()
        )
        | set(
            countries["Ethiopia"][feature].astype(str).unique()
        )
    )

    distributions = {}

    for country, df in countries.items():

        distributions[country] = (
            df[feature]
            .astype(str)
            .value_counts(normalize=True)
            .reindex(all_values, fill_value=0)
        )

    for country_1, country_2 in pairwise_countries:

        js_distance = jensenshannon(
            distributions[country_1].values,
            distributions[country_2].values,
            base=2
        )

        distribution_shift_results.append({
            "Feature": feature,
            "Comparison": f"{country_1} vs {country_2}",
            "JS_Distance": js_distance
        })


distribution_shift_df = pd.DataFrame(
    distribution_shift_results
)

distribution_shift_pivot = (
    distribution_shift_df
    .pivot(
        index="Feature",
        columns="Comparison",
        values="JS_Distance"
    )
    .round(4)
)

distribution_shift_pivot["Mean_JS_Distance"] = (
    distribution_shift_pivot.mean(axis=1)
).round(4)

distribution_shift_pivot = (
    distribution_shift_pivot
    .sort_values(
        "Mean_JS_Distance",
        ascending=False
    )
)

print("Cross-Country Predictor Distribution Shift")
print("Jensen-Shannon Distance")
print("=" * 90)

print(
    distribution_shift_pivot.to_string()
)

print("\nInterpretation guide:")
print("0.00 = identical distributions")
print("Larger values = stronger distributional difference")
print("Maximum possible value with base=2 = 1.00")

In [ ]:
# Cell 5: Summarize overall predictor distribution shift for each country pair and save the domain-shift results.

from pathlib import Path

pairwise_shift_summary = (
    distribution_shift_df
    .groupby("Comparison")["JS_Distance"]
    .agg(["mean", "median", "std", "max", "min"])
    .round(4)
    .sort_values("mean", ascending=False)
)

print("Overall Cross-Country Domain Shift")
print("=" * 75)
print(pairwise_shift_summary.to_string())

print("\nMost shifted feature in each country comparison:")
print("-" * 75)

for comparison in pairwise_countries:

    comparison_name = f"{comparison[0]} vs {comparison[1]}"

    subset = (
        distribution_shift_df[
            distribution_shift_df["Comparison"] == comparison_name
        ]
        .sort_values("JS_Distance", ascending=False)
    )

    top_row = subset.iloc[0]

    print(
        f"{comparison_name}: "
        f"{top_row['Feature']} "
        f"(JS = {top_row['JS_Distance']:.4f})"
    )


# Save for manuscript/reporting
table_dir = Path("../outputs/tables")
table_dir.mkdir(parents=True, exist_ok=True)

distribution_shift_pivot.to_csv(
    table_dir / "Table_Three_Country_Feature_Domain_Shift.csv"
)

pairwise_shift_summary.to_csv(
    table_dir / "Table_Three_Country_Overall_Domain_Shift.csv"
)

print("\nSaved:")
print("1. Table_Three_Country_Feature_Domain_Shift.csv")
print("2. Table_Three_Country_Overall_Domain_Shift.csv")


In [ ]:
# Cell 6: Prepare identical 11-feature datasets and country-specific class weights for six directional cross-country transfer experiments.

from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

transfer_features = common_features_3countries

transfer_categorical = [
    "day_common",
    "weather_common",
    "light_common",
    "surface_common",
    "junction_common"
]

transfer_data = {}

for country, df in countries.items():

    X = df[transfer_features].copy()
    y = df["severity_class"].copy()

    # CatBoost categorical variables as strings
    for col in transfer_categorical:
        X[col] = X[col].astype(str)

    # Calculate balanced weights from THIS training country
    counts = y.value_counts()
    n = len(y)
    k = len(counts)

    weights = {
        cls: n / (k * count)
        for cls, count in counts.items()
    }

    transfer_data[country] = {
        "X": X,
        "y": y,
        "weights": weights
    }

    print("\n" + "=" * 65)
    print(country)
    print("Samples:", len(y))
    print("Features:", X.shape[1])

    print("\nClass distribution:")
    print(y.value_counts())

    print("\nClass weights:")
    for cls, weight in weights.items():
        print(f"{cls}: {weight:.4f}")

In [ ]:
# Cell 7: Train cost-sensitive CatBoost models in each source country and evaluate all six directional cross-country transfers.

transfer_pairs = [
    ("UK", "France"),
    ("UK", "Ethiopia"),
    ("France", "UK"),
    ("France", "Ethiopia"),
    ("Ethiopia", "UK"),
    ("Ethiopia", "France")
]

transfer_results = []

for source_country, target_country in transfer_pairs:

    print("\n" + "=" * 80)
    print(f"Training: {source_country}  --->  Testing: {target_country}")
    print("=" * 80)

    X_train_transfer = transfer_data[source_country]["X"]
    y_train_transfer = transfer_data[source_country]["y"]

    X_test_transfer = transfer_data[target_country]["X"]
    y_test_transfer = transfer_data[target_country]["y"]

    source_weights = transfer_data[source_country]["weights"]

    # Train only on the source country
    transfer_model = CatBoostClassifier(
        iterations=500,
        depth=8,
        learning_rate=0.05,
        loss_function="MultiClass",
        random_seed=42,
        verbose=False,
        class_weights=source_weights
    )

    transfer_model.fit(
        X_train_transfer,
        y_train_transfer,
        cat_features=transfer_categorical
    )

    # Completely external prediction
    y_pred_transfer = transfer_model.predict(
        X_test_transfer
    ).ravel()

    report_transfer = classification_report(
        y_test_transfer,
        y_pred_transfer,
        labels=["Fatal", "Serious", "Slight"],
        output_dict=True,
        zero_division=0
    )

    result = {
        "Source": source_country,
        "Target": target_country,

        "Accuracy": accuracy_score(
            y_test_transfer,
            y_pred_transfer
        ),

        "Macro_Precision": precision_score(
            y_test_transfer,
            y_pred_transfer,
            average="macro",
            zero_division=0
        ),

        "Macro_Recall": recall_score(
            y_test_transfer,
            y_pred_transfer,
            average="macro",
            zero_division=0
        ),

        "Macro_F1": f1_score(
            y_test_transfer,
            y_pred_transfer,
            average="macro",
            zero_division=0
        ),

        "Fatal_Precision":
            report_transfer["Fatal"]["precision"],

        "Fatal_Recall":
            report_transfer["Fatal"]["recall"],

        "Fatal_F1":
            report_transfer["Fatal"]["f1-score"],

        "Serious_Precision":
            report_transfer["Serious"]["precision"],

        "Serious_Recall":
            report_transfer["Serious"]["recall"],

        "Serious_F1":
            report_transfer["Serious"]["f1-score"],

        "Slight_Precision":
            report_transfer["Slight"]["precision"],

        "Slight_Recall":
            report_transfer["Slight"]["recall"],

        "Slight_F1":
            report_transfer["Slight"]["f1-score"]
    }

    transfer_results.append(result)

    print(f"Accuracy:     {result['Accuracy']:.4f}")
    print(f"Macro F1:     {result['Macro_F1']:.4f}")
    print(f"Macro Recall: {result['Macro_Recall']:.4f}")

    print(
        f"Fatal Recall:   {result['Fatal_Recall']:.4f}"
    )

    print(
        f"Serious Recall: {result['Serious_Recall']:.4f}"
    )

    print(
        f"Slight Recall:  {result['Slight_Recall']:.4f}"
    )


transfer_results_df = pd.DataFrame(
    transfer_results
)

print("\n\nSix-Direction Cross-Country Transfer Results")
print("=" * 120)

display_columns = [
    "Source",
    "Target",
    "Accuracy",
    "Macro_Precision",
    "Macro_Recall",
    "Macro_F1",
    "Fatal_Recall",
    "Fatal_F1",
    "Serious_Recall",
    "Serious_F1",
    "Slight_Recall"
]

print(
    transfer_results_df[
        display_columns
    ].round(4).to_string(index=False)
)

In [ ]:
# Cell 8: Link pairwise feature-domain shift to directional cross-country transfer performance for exploratory analysis.

# Map symmetric country-pair JS distance to each directional transfer
pair_js = {
    frozenset(["UK", "France"]): 0.1113,
    frozenset(["UK", "Ethiopia"]): 0.1596,
    frozenset(["France", "Ethiopia"]): 0.2067
}

transfer_with_shift = transfer_results_df.copy()

transfer_with_shift["Mean_JS_Distance"] = transfer_with_shift.apply(
    lambda row: pair_js[frozenset([row["Source"], row["Target"]])],
    axis=1
)

analysis_columns = [
    "Source",
    "Target",
    "Mean_JS_Distance",
    "Accuracy",
    "Macro_F1",
    "Macro_Recall",
    "Fatal_Recall",
    "Serious_Recall",
    "Slight_Recall"
]

print("Domain Shift vs Cross-Country Transfer Performance")
print("=" * 110)

print(
    transfer_with_shift[
        analysis_columns
    ].round(4).to_string(index=False)
)

# Exploratory correlations only
correlation_metrics = [
    "Accuracy",
    "Macro_F1",
    "Macro_Recall",
    "Fatal_Recall",
    "Serious_Recall",
    "Slight_Recall"
]

correlation_results = []

for metric in correlation_metrics:

    pearson_corr = transfer_with_shift[
        ["Mean_JS_Distance", metric]
    ].corr(method="pearson").iloc[0, 1]

    spearman_corr = transfer_with_shift[
        ["Mean_JS_Distance", metric]
    ].corr(method="spearman").iloc[0, 1]

    correlation_results.append({
        "Metric": metric,
        "Pearson_r": pearson_corr,
        "Spearman_rho": spearman_corr
    })

correlation_results_df = pd.DataFrame(
    correlation_results
).round(4)

print("\nExploratory Correlation Between Domain Shift and Transfer Performance")
print("-" * 85)
print(correlation_results_df.to_string(index=False))

print("\nNote:")
print("These correlations are exploratory because only six directional transfers are available.")

In [ ]:
# Cell 9: Save the six-direction transfer results and exploratory domain-shift correlation tables for manuscript reporting.

from pathlib import Path

table_dir = Path("../outputs/tables")
table_dir.mkdir(parents=True, exist_ok=True)

transfer_results_df.to_csv(
    table_dir / "Table_Six_Direction_Cross_Country_Transfer.csv",
    index=False
)

transfer_with_shift.to_csv(
    table_dir / "Table_Domain_Shift_vs_Transfer_Performance.csv",
    index=False
)

correlation_results_df.to_csv(
    table_dir / "Table_Exploratory_Domain_Shift_Correlations.csv",
    index=False
)

print("Saved cross-country analysis tables:")
print("1. Table_Six_Direction_Cross_Country_Transfer.csv")
print("2. Table_Domain_Shift_vs_Transfer_Performance.csv")
print("3. Table_Exploratory_Domain_Shift_Correlations.csv")

In [ ]:
# Cell 10: Create publication-ready cross-country transferability heatmaps.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

country_order = ["UK", "France", "Ethiopia"]

metrics_for_heatmap = {
    "Macro F1": "Macro_F1",
    "Macro Recall": "Macro_Recall",
    "Fatal Recall": "Fatal_Recall"
}

fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 4.8)
)

for ax, (title, metric) in zip(
    axes,
    metrics_for_heatmap.items()
):

    matrix = pd.DataFrame(
        np.nan,
        index=country_order,
        columns=country_order
    )

    for _, row in transfer_results_df.iterrows():

        matrix.loc[
            row["Source"],
            row["Target"]
        ] = row[metric]

    image = ax.imshow(
        matrix.values,
        vmin=0,
        vmax=0.75
    )

    ax.set_xticks(
        range(len(country_order))
    )

    ax.set_xticklabels(
        country_order
    )

    ax.set_yticks(
        range(len(country_order))
    )

    ax.set_yticklabels(
        country_order
    )

    ax.set_xlabel("Target country")
    ax.set_ylabel("Training country")
    ax.set_title(title)

    # Write numerical values inside cells
    for i in range(len(country_order)):
        for j in range(len(country_order)):

            value = matrix.iloc[i, j]

            if pd.notna(value):

                ax.text(
                    j,
                    i,
                    f"{value:.3f}",
                    ha="center",
                    va="center",
                    fontsize=11,
                    fontweight="bold"
                )

            else:

                ax.text(
                    j,
                    i,
                    "—",
                    ha="center",
                    va="center",
                    fontsize=12
                )

fig.colorbar(
    image,
    ax=axes,
    fraction=0.025,
    pad=0.04,
    label="Performance"
)

fig.suptitle(
    "Cross-Country Transportability of Crash-Severity Models",
    fontsize=15,
    y=1.02
)

plt.tight_layout()

plt.savefig(
    "../outputs/figures/Figure_Cross_Country_Transferability.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Cell 11: Plot the relationship between cross-country domain shift and transfer performance for Macro Recall and Fatal Recall.

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))

# Plot Macro Recall
ax.scatter(
    transfer_with_shift["Mean_JS_Distance"],
    transfer_with_shift["Macro_Recall"],
    s=90,
    label="Macro Recall"
)

# Plot Fatal Recall
ax.scatter(
    transfer_with_shift["Mean_JS_Distance"],
    transfer_with_shift["Fatal_Recall"],
    s=90,
    marker="s",
    label="Fatal Recall"
)

# Add directional transfer labels
for _, row in transfer_with_shift.iterrows():

    label = f"{row['Source']}→{row['Target']}"

    ax.annotate(
        label,
        (
            row["Mean_JS_Distance"],
            row["Macro_Recall"]
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9
    )

    ax.annotate(
        label,
        (
            row["Mean_JS_Distance"],
            row["Fatal_Recall"]
        ),
        xytext=(5, -12),
        textcoords="offset points",
        fontsize=9
    )

ax.set_xlabel("Mean Jensen–Shannon Distance")
ax.set_ylabel("Recall")
ax.set_title(
    "Domain Shift and Cross-Country Crash-Severity Transfer Performance"
)

ax.set_ylim(0, 0.55)

ax.legend()

plt.tight_layout()

plt.savefig(
    "../outputs/figures/Figure_Domain_Shift_vs_Transfer_Performance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Cell 12: Visualize Macro-F1 across all six directional cross-country transfer experiments.

transfer_plot_df = transfer_results_df.copy()

transfer_plot_df["Transfer"] = (
    transfer_plot_df["Source"]
    + " → "
    + transfer_plot_df["Target"]
)

transfer_plot_df = transfer_plot_df.sort_values(
    "Macro_F1",
    ascending=False
)

fig, ax = plt.subplots(figsize=(9, 5.5))

bars = ax.bar(
    transfer_plot_df["Transfer"],
    transfer_plot_df["Macro_F1"]
)

ax.set_title(
    "Macro-F1 Across Six Cross-Country Transfer Experiments"
)

ax.set_xlabel("Training → External Test Country")
ax.set_ylabel("Macro F1-score")

ax.set_ylim(
    0,
    max(transfer_plot_df["Macro_F1"]) + 0.08
)

plt.xticks(
    rotation=30,
    ha="right"
)

for bar, value in zip(
    bars,
    transfer_plot_df["Macro_F1"]
):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    "../outputs/figures/Figure_Six_Direction_Macro_F1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Cell 13: Rebuild the final UK cost-sensitive CatBoost model independently in Notebook 06 for SHAP analysis.

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

shap_features = [
    "hour",
    "day_common",
    "weather_common",
    "light_common",
    "surface_common",
    "junction_common",
    "vehicle_count",
    "has_motorcycle",
    "has_heavy_vehicle",
    "has_public_transport",
    "has_two_wheeler"
]

shap_categorical = [
    "day_common",
    "weather_common",
    "light_common",
    "surface_common",
    "junction_common"
]

X_uk_shap = uk_df[shap_features].copy()
y_uk_shap = uk_df["severity_class"].copy()

X_train_uk_shap, X_test_uk_shap, y_train_uk_shap, y_test_uk_shap = train_test_split(
    X_uk_shap,
    y_uk_shap,
    test_size=0.20,
    random_state=42,
    stratify=y_uk_shap
)

# Balanced class weights calculated only from UK training data
class_counts_shap = y_train_uk_shap.value_counts()
n_train_shap = len(y_train_uk_shap)
n_classes_shap = len(class_counts_shap)

balanced_weights_shap = {
    cls: n_train_shap / (n_classes_shap * count)
    for cls, count in class_counts_shap.items()
}

selected_alpha = 0.75

final_weights_shap = {
    cls: weight ** selected_alpha
    for cls, weight in balanced_weights_shap.items()
}

final_catboost_model_shap = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False,
    class_weights=final_weights_shap
)

final_catboost_model_shap.fit(
    X_train_uk_shap,
    y_train_uk_shap,
    cat_features=shap_categorical
)

print("Final UK CatBoost model rebuilt successfully for SHAP.")
print("Selected alpha:", selected_alpha)

print("\nClass weights:")
for cls, weight in final_weights_shap.items():
    print(f"{cls}: {weight:.4f}")

print("\nTraining shape:", X_train_uk_shap.shape)
print("Test shape:", X_test_uk_shap.shape)
print("Model classes:", final_catboost_model_shap.classes_)